# Lean Agent Playground

Wires up a smolagents `CodeAgent` with a hand-picked list of tools, runs it, and saves the run logs to disk.

Requires `OPENAI_API_KEY` in `.env`.

In [22]:
from smolagents import CodeAgent, ToolCallingAgent, OpenAIServerModel

from lean_agent import get_settings, save_run
from lean_agent.tools import (
    caesar_cipher,
    compound_interest,
    fibonacci,
    get_current_time,
    is_palindrome,
    is_prime,
    reverse_words,
    roll_dice,
    temperature_convert,
    word_count,
)
from lean_agent.putnambench_tools import get_putnambench_problems, get_problem_statement

settings = get_settings()
print(f"model:   {settings.model_id}")
print(f"log dir: {settings.log_dir}")

model:   gpt-5.4-nano
log dir: /Users/nhantruong/Coding/lean-agent/logs


---

## 1. Build the agent

Three things to notice:

1. **`tools=[...]`** &mdash; an explicit list. Swap tools in/out to change what the agent can do.
2. **`max_steps`** &mdash; hard cap on the number of reasoning iterations. Useful guard against runaway loops.
3. **`instructions`** &mdash; short, high-priority guidance prepended to the system prompt. Keep it tight; it competes with the rest of smolagents' built-in prompt.

In [23]:
model = OpenAIServerModel(
    model_id=settings.model_id,
    api_key=settings.api_key,
    api_base=settings.api_base,  # None -> hosted OpenAI; set to e.g. http://localhost:8000/v1 for vLLM
)

tools = [
    get_current_time,
    word_count,
    fibonacci,
    compound_interest,
    is_prime,
    is_palindrome,
    temperature_convert,
    roll_dice,
    caesar_cipher,
    reverse_words,
    get_putnambench_problems,
    get_problem_statement,
]

agent = ToolCallingAgent(
    tools=tools,
    model=model,
    max_steps=12,
    instructions="Prefer using a tool when one fits the question. Be concise.",
)

print(f"{len(tools)} tools wired up:")
for t in tools:
    print(f"  - {t.name}")

12 tools wired up:
  - get_current_time
  - word_count
  - fibonacci
  - compound_interest
  - is_prime
  - is_palindrome
  - temperature_convert
  - roll_dice
  - caesar_cipher
  - reverse_words
  - get_putnambench_problems
  - get_problem_statement


---

## 2. Run a prompt that exercises several tools

In [24]:
task = (
    "Get number theory Putnam problems from the last 10 years."
)

answer = agent.run(task)

print("\n--- final answer ---")
print(answer)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Get number theory Putnam problems from the last 10 years.                                                       │
│                                                                                                                 │
╰─ OpenAIModel - gpt-5.4-nano ────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Error while generating output:
Error code: 401 - {'error': {'message': 'Incorrect API key provided: 
sk-proj-***********************************************************************************************************
*********************************************M00A. You can find your API key at 
https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 
'param': None}, 'status': 401}

[Step 1: Duration 0.80 seconds]

AgentGenerationError: Error while generating output:
Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-proj-********************************************************************************************************************************************************M00A. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}

---

## 3. Save the run logs

`save_run(agent, answer)` writes two files into a fresh per-run directory under `logs/`. Pass `run_id="my-experiment"` if you want to label this run; otherwise it auto-generates one.

- `run.json` &mdash; `{manifest, prompt, answer, logs}` with per-step + total token usage.
- `transcript.yaml` &mdash; sanitized `system / user / assistant / tool-*` linear chat view.

In [8]:
run_dir = save_run(agent, answer)
print(f"logs saved to: {run_dir}")
for f in sorted(run_dir.iterdir()):

    print(f"  {f.name}  ({f.stat().st_size} bytes)")

logs saved to: /Users/nhantruong/Coding/lean-agent/logs/20260528T003911Z-d4bb7d
  run.json  (15806 bytes)
  transcript.yaml  (9214 bytes)
